In [0]:
catalog_name = "pharma_demo_catalog"
volume_path = f"/Volumes/{catalog_name}/bronze/raw_data/"

In [0]:
from pyspark.sql.functions import col, lit, round, current_timestamp, sha2, concat_ws, when

df_iot_clean = spark.table(f"{catalog_name}.bronze.iot_sensor_raw") \
    .filter(col("is_null_record") == False) \
    .withColumn("parameter_value", col("parameter_value").cast("decimal(10,2)")) \
    .drop("is_null_record")


df_iot_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.iot_sensor_cleaned")


df_mes_current = spark.table(f"{catalog_name}.bronze.mes_batch_raw").filter(col("is_current") == True)


df_erp_silver = df_mes_current.withColumn(
    "erp_vial_count", (col("actual_quantity") - col("rejected_quantity"))
).withColumn(
    "erp_yield_pct", round((col("erp_vial_count") * 100.0) / col("planned_quantity"), 2)
).withColumn(
    "row_integrity_hash", sha2(concat_ws("||", col("batch_id"), col("erp_vial_count")), 256) 
)


df_erp_final = df_erp_silver.withColumn(
    "performance_status", when(col("erp_yield_pct") < 92, lit("UNDERPERFORMING")).otherwise(lit("OPTIMAL"))
).withColumn("transformation_ts", current_timestamp())

df_erp_final.select(
    "batch_id", 
    "product_code", 
    "erp_vial_count", 
    "erp_yield_pct", 
    "performance_status", 
    "row_integrity_hash", 
    "transformation_ts"
).write.format("delta") \
 .mode("overwrite") \ 
 .option("overwriteSchema", "true") \
 .saveAsTable(f"{catalog_name}.silver.erp_inventory_silver")


spark.read.csv(f"/Volumes/{catalog_name}/bronze/raw_data/silver_lims_quality_results.csv", header=True, inferSchema=True) \
    .write.format("delta").mode("overwrite").saveAsTable(f"{catalog_name}.silver.lims_quality_conformed")



Silver Layer Advanced: Precision and Security hashing applied.
